In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [27]:
kc_house_data = pd.read_csv('./data/kc_house_data.csv')
kc_house_data.head(5)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


Bias refers to the error introduced by approximating a complex real-world problem with a simplified model, while variance refers to the model's sensitivity to fluctuations in the training data. A linear regression model has high bias and low variance; it makes strong assumptions about the data (linearity) but is stable across different datasets. If these strong assumptions are not correct, there will be places where it systematically overestimates or underestimates. In contrast, a decision tree model has low bias and high variance;it can capture complex patterns but is prone to overfitting, especially if deep and unpruned. This means that it can start to memorize the training data rather than capturing patterns that generalize.

Fit a linear regression model to the housing data, using sqft_living to predict price. Check the mean squared error on the training data and the test data.

In [ ]:
features = ['sqft_living']
X = kc_house_data[features]
y = kc_house_data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

reg = LinearRegression().fit(X_train, y_train)

In [28]:
y_pred_train = reg.predict(X_train)
print(f'MSE for training data: {mean_squared_error(y_train, y_pred_train):,}')

MSE for training data: 49,021,097,271.69459


In [29]:
y_pred_test = reg.predict(X_test)
print(f'MSE for testing data: {mean_squared_error(y_test, y_pred_test):,}')

MSE for testing data: 69,489,188,925.29305


Repeat this but with a DecisionTreeRegresor. Again check the mean squared error on the training data and the test data. How does what you see differ from the linear regression model?

In [30]:
dt_reg = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)

In [31]:
y_pred_train = dt_reg.predict(X_train)
print(f'MSE for training data: {mean_squared_error(y_train, y_pred_train):,}')

MSE for training data: 49,021,097,271.69459


In [32]:
y_pred_test = dt_reg.predict(X_test)
print(f'MSE for testing data: {mean_squared_error(y_test, y_pred_test):,}')

MSE for testing data: 69,489,188,925.29305


The MSE for training vs testing data is very close for the linear regression model. For the decision tree regressor, the MSE is much higher for the testing data versus the training data, indicating the model might be overfit to the training data.

One way of avoiding overfitting is by restricting the flexibility of the model. We can do this with a decision tree by restricting the number of splits that it can perform.

Fit a DecisionTreeRegressor where you restrict the max_depth to 5. Again check the mean squared error on the training data and the test data. What do you notice now?

In [33]:
dt_reg_limited = DecisionTreeRegressor(random_state=0, max_depth=5).fit(X_train, y_train)

In [34]:
y_pred_train = dt_reg_limited.predict(X_train)
print(f'MSE for training data: {mean_squared_error(y_train, y_pred_train):,}')

MSE for training data: 59,193,410,119.57668


In [35]:
y_pred_test = dt_reg_limited.predict(X_test)
print(f'MSE for testing data: {mean_squared_error(y_test, y_pred_test):,}')

MSE for testing data: 62,806,766,293.83895


Setting a max depth helped bring the MSE of the testing data closer to the MSE of the training data.

When working with machine learning models, we often have to balance bias and variance. This is called the bias-variance tradeoff. One method of this is through regularization, where we restrict the complexity of the model, adding some bias but reducing the variance, which can lead to a lower mean squared error on the test set.

Lasso and ridge regression do this by adding a penalty term based on the size of the coefficients. Smaller coefficients means that the model has less flexibility. The neat thing about these types of models is that they determine how to allocate the coefficients automatically as part of the model fitting process, so we can start with a large set of potential predictors and allow the model fitting to determine which ones to focus on.

For the next part of the exercise, we'll see how we can add complexity to our model but control the complexity through regularization.

So far, we've only been predicting based off of the square footage of living space. Fit a new linear regression model using all variables besides id, date, price, and zipcode. How well does this model perform on the test set compared to the model with just square footage of living space?

In [36]:
features = [col for col in kc_house_data.columns if col not in ['id', 'date', 'price', 'zipcode']]
X = kc_house_data[features]
y = kc_house_data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

reg = LinearRegression().fit(X_train, y_train)

In [38]:
y_pred_test = reg.predict(X_test)
print(f'MSE for testing data: {mean_squared_error(y_test, y_pred_test):,}')

MSE for testing data: 43,690,646,677.91591


The model using all variables except id, date, price, and zipcode makes much better predictions than the one with just square footage (MSE of 43,690,646,677 vs 69,489,188,925)

Try fitting a lasso and ridge model. Becuase lasso and ridge have penalty terms based on the size of the coefficients, and the size of the coefficients depends on the scale of the variable, you'll want to scale the features first so that they are on comparable scales. Create a Pipeline object where the first step is applying a StandardScaler and the second step is either a lasso or ridge model. Because these models have a hyperparameter controlling regularization strength, you'll want to use the LassoCV and RidgeCV models, which will select values for the regularization strength using cross-validation.

In [43]:
lasso = Pipeline(
    steps=[("standard_scaler", StandardScaler()), ("lasso_cv", LassoCV(cv=5, random_state=0))]
)

features = [col for col in kc_house_data.columns if col not in ['id', 'date', 'price', 'zipcode']]
X = kc_house_data[features]
y = kc_house_data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

lasso.fit(X_train, y_train)

Pipeline(steps=[('standard_scaler', StandardScaler()),
                ('lasso_cv', LassoCV(cv=5, random_state=0))])

In [44]:
y_pred_test = lasso.predict(X_test)
print(f'MSE for testing data: {mean_squared_error(y_test, y_pred_test):,}')

MSE for testing data: 43,693,367,645.13081


You likely didn't see much difference between the regular linear regression model and the lasso or ridge model. Let's see what happens if we add more complexity through feature interactions. We can capture more complex relationships between the predictor variables and the target variable by multiplying the predictors together before fitting the model. For example, the interaction between sqft_living and bedrooms will let the model capture if the impact of square footage depends on the number of bedrooms.